# Isobaric chain A = 128: energy vs Z, decay modes and β end-point energies

**Given** atomic masses (in u) for $_{50}$Sn … $_{56}$Ba, all with A = 128.

**Tasks**
1. Draw a diagram of energy (mass) versus atomic number Z.
2. Label the possible decay modes and justify them.
3. Find, where applicable, the maximum kinetic energy of the β particles.
4. Decide which nuclei are stable.

Because the masses are **atomic** masses (they include the electrons), the Q-values are:

| Decay | Condition / Q-value |
|---|---|
| β⁻ : $Z \to Z+1$ | $Q_{\beta^-} = [M(Z) - M(Z+1)]\,c^2 > 0$ |
| Electron capture (EC): $Z \to Z-1$ | $Q_{EC} = [M(Z) - M(Z-1)]\,c^2 > 0$ |
| β⁺ : $Z \to Z-1$ | $Q_{\beta^+} = [M(Z) - M(Z-1) - 2m_e]\,c^2 > 0$ |
| Double β⁻ : $Z \to Z+2$ | $Q_{\beta\beta} = [M(Z) - M(Z+2)]\,c^2 > 0$ |

The daughter nucleus is ~10⁵ times heavier than the electron, so its recoil energy is negligible: the **maximum kinetic energy of the β particle ≈ Q** (reached when the neutrino carries away almost no energy).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

u_to_MeV = 931.494     # 1 u c^2 in MeV
me_u     = 0.000548580 # electron mass in u
me_MeV   = me_u * u_to_MeV

data = {  # Z: (symbol, atomic mass in u)
    50: ("Sn", 127.91047),
    51: ("Sb", 127.90907),
    52: ("Te", 127.90447),
    53: ("I",  127.90583),
    54: ("Xe", 127.90354),
    55: ("Cs", 127.90776),
    56: ("Ba", 127.90851),
}
Z    = np.array(sorted(data))
sym  = [data[z][0] for z in Z]
M    = np.array([data[z][1] for z in Z])
Mmin = M.min()
E    = (M - Mmin) * u_to_MeV          # energy relative to the lightest isobar, MeV

for z, s, m, e in zip(Z, sym, M, E):
    parity = ("even" if z % 2 == 0 else "odd") + "-" + ("even" if (128 - z) % 2 == 0 else "odd")
    print(f"{z:3d} {s:3s} M = {m:.5f} u   E - E_min = {e:6.3f} MeV   ({parity}, N = {128 - z})")

## Q-values for every allowed single-step decay

In [ ]:
decays = {}   # (parent Z) -> list of (mode, daughter Z, Q in MeV)

for z in Z:
    m = data[z][1]
    modes = []
    # beta-minus to Z+1
    if z + 1 in data:
        Q = (m - data[z + 1][1]) * u_to_MeV
        if Q > 0:
            modes.append(("β⁻", z + 1, Q))
    # EC and beta-plus to Z-1
    if z - 1 in data:
        Qec = (m - data[z - 1][1]) * u_to_MeV
        if Qec > 0:
            modes.append(("EC", z - 1, Qec))
            Qbp = Qec - 2 * me_MeV
            if Qbp > 0:
                modes.append(("β⁺", z - 1, Qbp))
    decays[z] = modes

print(f"2 m_e c^2 = {2*me_MeV:.3f} MeV\n")
for z in Z:
    s = data[z][0]
    if not decays[z]:
        print(f"{s}-128 : no single-β decay energetically allowed")
    for mode, zd, Q in decays[z]:
        tail = "   -> max β kinetic energy ≈ Q" if mode in ("β⁻", "β⁺") else "   (no β emitted; neutrino + X-rays)"
        print(f"{s}-128 --{mode:2s}--> {data[zd][0]}-128 : Q = {Q:6.3f} MeV{tail}")

## Double-β decay check for nuclei that are stable against single β decay

In [ ]:
single_stable = [z for z in Z if not decays[z]]
for z in single_stable:
    s = data[z][0]
    for dz, label in [(+2, "2β⁻"), (-2, "2EC")]:
        if z + dz in data:
            Q = (data[z][1] - data[z + dz][1]) * u_to_MeV
            verdict = "allowed (but second-order weak process, T½ ~ 10²⁴ y)" if Q > 0 else "forbidden"
            print(f"{s}-128 --{label}--> {data[z+dz][0]}-128 : Q = {Q:6.3f} MeV  -> {verdict}")

## Energy vs Z diagram

Odd-A chains lie on a single parabola. For **even A** the pairing term splits the points onto **two parabolas**: even-even nuclei (even Z here, since N = 128 − Z is then even) lie lower, odd-odd nuclei (odd Z) lie higher. We fit a parabola through each set as a guide to the eye.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

even = Z % 2 == 0
odd  = ~even
zz = np.linspace(49.6, 56.4, 300)
for mask, col, lab in [(even, "tab:blue", "even-even (even Z)"), (odd, "tab:red", "odd-odd (odd Z)")]:
    c = np.polyfit(Z[mask], E[mask], 2)
    ax.plot(zz, np.polyval(c, zz), "--", color=col, alpha=0.5)
    ax.plot(Z[mask], E[mask], "o", ms=10, color=col, label=lab)

for z, s, e in zip(Z, sym, E):
    ax.annotate(f"$^{{128}}${s}", (z, e), textcoords="offset points", xytext=(0, 12),
                ha="center", fontsize=11, fontweight="bold")

colours = {"β⁻": "tab:green", "β⁺": "tab:purple", "EC": "tab:orange"}
for z in Z:
    modes = decays[z]
    has_bp = any(m == "β⁺" for m, _, _ in modes)
    for mode, zd, Q in modes:
        if mode == "EC" and has_bp:      # draw one arrow for EC/β⁺, label both
            continue
        e0, e1 = E[Z == z][0], E[Z == zd][0]
        ax.annotate("", xy=(zd, e1 + 0.08), xytext=(z, e0 - 0.08),
                    arrowprops=dict(arrowstyle="-|>", color=colours[mode], lw=1.8))
        if mode == "β⁺":
            Qec = [q for m, d, q in modes if m == "EC"][0]
            text = f"EC/β⁺\nQ_EC={Qec:.2f}\nT_max(β⁺)={Q:.2f}"
        elif mode == "EC":
            text = f"EC only\nQ={Q:.2f}"
        else:
            text = f"β⁻\nT_max={Q:.2f}"
        xm, ym = (z + zd) / 2, (e0 + e1) / 2
        dx, dy = {53: (-0.15, 0.65)}.get(z, (0, 0))   # nudge labels that would cover a point
        xm, ym = xm + dx, ym + dy
        ax.text(xm, ym, text, color=colours[mode], fontsize=8.5, ha="center", va="center",
                bbox=dict(fc="white", ec="none", alpha=0.85))

# double-beta of Te-128
zt, zx = 52, 54
ax.annotate("", xy=(zx, E[Z == zx][0] + 0.05), xytext=(zt, E[Z == zt][0] + 0.05),
            arrowprops=dict(arrowstyle="-|>", color="grey", lw=1.2, ls=":", connectionstyle="arc3,rad=0.25"))
Qbb = (data[52][1] - data[54][1]) * u_to_MeV
ax.text(53, -0.68, f"2β⁻ (very slow)\nQ={Qbb:.2f}", color="grey", ha="center", fontsize=8.5)

ax.set_xlabel("Atomic number Z")
ax.set_ylabel(r"$(M - M_{\min})\,c^2$  [MeV]   (relative to $^{128}$Xe)")
ax.set_title("Isobaric chain A = 128 (energies in MeV)")
ax.set_xticks(Z)
ax.set_xticklabels([f"{z}\n{s}" for z, s in zip(Z, sym)])
ax.set_ylim(-1.0, E.max() + 0.8)
ax.grid(alpha=0.3)
ax.legend(loc="upper center")
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
print(f"{'Parent':8s}{'Mode':8s}{'Daughter':10s}{'Q (MeV)':>9s}{'T_max β (MeV)':>15s}")
print("-" * 50)
for z in Z:
    for mode, zd, Q in decays[z]:
        tmax = f"{Q:.3f}" if mode in ("β⁻", "β⁺") else "—"
        print(f"{data[z][0]+'-128':8s}{mode:8s}{data[zd][0]+'-128':10s}{Q:9.3f}{tmax:>15s}")
print("\nStable against single β decay:", ", ".join(data[z][0] + "-128" for z in single_stable))

## Discussion

**Decay modes**
- **¹²⁸Sn → ¹²⁸Sb → ¹²⁸Te**: on the neutron-rich side every nucleus is heavier than its Z+1 neighbour, so they decay by **β⁻**. End-point energies ≈ 1.30 MeV and 4.29 MeV.
- **¹²⁸I** (odd-odd) sits on the upper parabola *above both* of its even-even neighbours, so it can decay **both ways**: β⁻ to ¹²⁸Xe (T_max ≈ 2.13 MeV) and EC/β⁺ to ¹²⁸Te (Q_EC ≈ 1.27 MeV; since this exceeds 2m_ec² = 1.022 MeV, β⁺ is just possible with T_max ≈ 0.25 MeV). This is typical of odd-odd nuclei in even-A chains.
- **¹²⁸Cs** → ¹²⁸Xe by **EC and β⁺** (Q_EC ≈ 3.93 MeV, T_max(β⁺) ≈ 2.91 MeV).
- **¹²⁸Ba** → ¹²⁸Cs: Q_EC ≈ 0.70 MeV < 1.022 MeV, so **only electron capture** is possible; β⁺ is energetically forbidden.

**Stable nuclei**
Because of the pairing energy, an even-A chain can have **more than one** β-stable nucleus. Both even-even nuclei **¹²⁸Te** and **¹²⁸Xe** are lower in energy than their odd-odd neighbours ¹²⁸Sb, ¹²⁸I and ¹²⁸Cs, so neither can undergo a single β decay. ¹²⁸Xe is the absolute minimum. ¹²⁸Te could in principle reach ¹²⁸Xe by **double β⁻ decay** (Q ≈ 0.87 MeV), but this second-order process is so slow (measured T½ ~ 10²⁴ y) that ¹²⁸Te is effectively stable.

→ Expected stable: **¹²⁸Te and ¹²⁸Xe**.